# BYOS: Sustainability Reports + Judge+Critic Evaluation (H3)

This notebook covers the final two stages of the pipeline:
1. **Report Generation** — aggregate per-user emissions stats and call GPT-4.1 to write a sustainability report
2. **Judge+Critic Agent** — evaluate each report's factual accuracy using a two-pass LLM loop
3. **H3 Evaluation** — compare Judge+Critic vs. single-prompt judge using Cohen's κ against human labels

**Prerequisites:** run `modeling.ipynb` first — this notebook loads `models/lgbm_mode_classifier.pkl`
and reconstructs `em_df` from the saved model and feature cache.

**Requires:** `OPENAI_API_KEY` set in `.env`

## 0. Setup

In [ ]:
import sys
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sys.path.insert(0, str(Path.cwd()))
load_dotenv()

# ── Load saved model artifact ─────────────────────────────────────────────────
MODEL_PATH = Path('../models/lgbm_mode_classifier.pkl')
with open(MODEL_PATH, 'rb') as f:
    artifact = pickle.load(f)

model        = artifact['model']
FEATURE_COLS = artifact['feature_cols']
le           = artifact['label_encoder']
print(f'Model loaded — macro-F1: {artifact["macro_f1"]:.3f}')
print(f'Features: {FEATURE_COLS}')
print(f'Classes:  {list(le.classes_)}')

# ── Reconstruct test split (same seed as modeling.ipynb) ─────────────────────
from sklearn.model_selection import StratifiedGroupKFold
from features import build_feature_dataset
import kagglehub

path    = kagglehub.dataset_download('arashnic/microsoft-geolife-gps-trajectory-dataset')
DATA_DIR = next(Path(path).rglob('Data'))
feature_df = build_feature_dataset(DATA_DIR)
feature_df['mode'] = pd.Categorical(feature_df['mode'], categories=sorted({'walk','bike','bus','car'}))

df = feature_df.dropna(subset=FEATURE_COLS).copy()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df, y=df['mode'], groups=df['user']))
test_df = df.iloc[test_idx].copy()

# ── Build em_df with predictions ─────────────────────────────────────────────
EMISSION_FACTORS = {'car': 170, 'bus': 89, 'bike': 0, 'walk': 0}
y_pred = model.predict(test_df[FEATURE_COLS])
em_df  = test_df.copy()
em_df['pred_mode']   = le.inverse_transform(y_pred)
em_df['dist_km']     = em_df['distance_total_m'] / 1000
em_df['dist_km_adj'] = em_df['dist_km'] * 0.5
em_df['co2_g']       = em_df['pred_mode'].map(EMISSION_FACTORS).fillna(0) * em_df['dist_km_adj']

print(f'\nTest users: {sorted(em_df["user"].unique())}')
print(f'Windows:    {len(em_df):,}')


---
## 1. Sustainability Report Generation

For each test user we:
1. Aggregate emissions stats from `em_df` into a structured dict
2. Call GPT-4.1 with the stats and a factuality constraint
3. Save the report alongside the raw stats for Judge+Critic evaluation

**Factuality constraint:** every number in the report must come directly from the data.
This sets up verifiable claims for H3.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
MODEL  = 'gpt-4.1'


def build_user_stats(user: str, em_df) -> dict:
    """Aggregate emissions stats for one user."""
    u        = em_df[em_df['user'] == user]
    mode_km  = u.groupby('pred_mode')['dist_km_adj'].sum().round(2).to_dict()
    mode_co2 = u.groupby('pred_mode')['co2_g'].sum().div(1000).round(2).to_dict()
    total_co2 = round(sum(mode_co2.values()), 2)
    car_rows  = u[u['pred_mode'] == 'car']
    car_co2   = mode_co2.get('car', 0)
    return {
        'user':            user,
        'mode_km':         mode_km,
        'mode_co2_kg':     mode_co2,
        'total_co2_kg':    total_co2,
        'car_pct_of_co2':  round(100 * car_co2 / total_co2, 1) if total_co2 else 0,
    }


def generate_report(stats: dict) -> str:
    """Call GPT-4.1 to generate a sustainability report from user stats."""
    prompt = (
        'You are a sustainability analyst writing a personal mobility report.\n'
        'Rules:\n'
        '  1. Write exactly 4-6 sentences.\n'
        '  2. Every number you state must come directly from the data below — '
        'do not round, estimate, or invent figures.\n'
        '  3. End with one concrete, actionable recommendation based on the data.\n\n'
        f'User mobility data:\n{json.dumps(stats, indent=2)}\n\n'
        'Report:'
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=350,
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


reports = {}
for user in sorted(em_df['user'].unique()):
    stats  = build_user_stats(user, em_df)
    report = generate_report(stats)
    reports[user] = {'stats': stats, 'report': report}
    print(f'\n--- User {user} ---')
    print(f'Total CO₂: {stats["total_co2_kg"]} kg | Car: {stats["car_pct_of_co2"]}%')
    print(report)

print(f'\n{len(reports)} reports generated.')


In [ ]:
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

out_path = REPORTS_DIR / 'user_reports.json'
with open(out_path, 'w') as f:
    json.dump(reports, f, indent=2)

print(f'Saved {len(reports)} reports → {out_path}')
print('Next: section 2 — Judge+Critic evaluation')


---
## 2. Judge+Critic Agent (H3)

**Goal:** Evaluate each report's factual accuracy using a two-pass LLM loop, then
compare against a single-prompt baseline to test whether the loop improves agreement
with human labels (Cohen's κ).

**Loop:**
```
Report + raw stats
    ↓  Judge pass — verify each numeric claim
Judge verdicts
    ↓  Critic pass — flag weak justifications
Critic flags
    ↓  Judge revises flagged verdicts
Final verdicts: {correct, incorrect, unverifiable} per claim
```

**Baseline:** single Claude prompt with same inputs, no Critic pass.

In [ ]:
import anthropic

claude = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
JUDGE_MODEL = 'claude-sonnet-4-6'


def judge_report(stats: dict, report: str) -> str:
    """Judge pass: verify each numeric claim in the report."""
    prompt = (
        'You are a fact-checker for sustainability reports.\n'
        'Given the raw data and the report, identify each numeric claim in the report\n'
        'and label it: correct / incorrect / unverifiable.\n'
        'Cite the specific data value that supports or contradicts each claim.\n\n'
        f'Raw data:\n{json.dumps(stats, indent=2)}\n\n'
        f'Report:\n{report}\n\n'
        'List each claim and verdict:'
    )
    return claude.messages.create(
        model=JUDGE_MODEL, max_tokens=600,
        messages=[{'role': 'user', 'content': prompt}]
    ).content[0].text


def critic_pass(judge_output: str) -> str:
    """Critic pass: identify weak justifications in the Judge's assessment."""
    prompt = (
        'You are reviewing a fact-checker\'s assessment of a sustainability report.\n'
        'Identify any claims the fact-checker accepted too easily, any verdicts\n'
        'that lack clear data support, or any missed claims.\n'
        'Be specific about which verdicts need revision and why.\n\n'
        f'Fact-checker assessment:\n{judge_output}\n\n'
        'Flags for revision:'
    )
    return claude.messages.create(
        model=JUDGE_MODEL, max_tokens=400,
        messages=[{'role': 'user', 'content': prompt}]
    ).content[0].text


def judge_revised(stats: dict, report: str, critic_flags: str) -> str:
    """Judge revises verdicts based on Critic flags."""
    prompt = (
        'You previously assessed a sustainability report. A critic has flagged issues\n'
        'with your assessment. Revise your verdicts where the critic raises valid points.\n\n'
        f'Raw data:\n{json.dumps(stats, indent=2)}\n\n'
        f'Report:\n{report}\n\n'
        f'Critic flags:\n{critic_flags}\n\n'
        'Final revised verdicts (correct / incorrect / unverifiable per claim):'
    )
    return claude.messages.create(
        model=JUDGE_MODEL, max_tokens=600,
        messages=[{'role': 'user', 'content': prompt}]
    ).content[0].text


def baseline_judge(stats: dict, report: str) -> str:
    """Single-prompt baseline — no Critic pass."""
    prompt = (
        'Given the data and the report, is each numeric claim correct, incorrect,\n'
        'or unverifiable? List each claim with its verdict.\n\n'
        f'Data:\n{json.dumps(stats, indent=2)}\n\n'
        f'Report:\n{report}\n\n'
        'Verdicts:'
    )
    return claude.messages.create(
        model=JUDGE_MODEL, max_tokens=600,
        messages=[{'role': 'user', 'content': prompt}]
    ).content[0].text


# ── Run Judge+Critic loop on all reports ──────────────────────────────────────
eval_results = {}
for user, data in reports.items():
    stats  = data['stats']
    report = data['report']

    j1      = judge_report(stats, report)
    critic  = critic_pass(j1)
    j_final = judge_revised(stats, report, critic)
    baseline = baseline_judge(stats, report)

    eval_results[user] = {
        'stats':          stats,
        'report':         report,
        'judge_initial':  j1,
        'critic':         critic,
        'judge_final':    j_final,
        'baseline_judge': baseline,
    }
    print(f'\n=== User {user} ===')
    print(f'--- Baseline judge ---\n{baseline[:300]}')
    print(f'--- Final (loop) judge ---\n{j_final[:300]}')

# Save for H3 Cohen's κ computation
out_path = Path('../reports/eval_results.json')
with open(out_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'\nSaved eval results → {out_path}')
print('Next: hand-label claims → compute Cohen\'s κ (section 3)')


---
## 3. H3 Evaluation — Cohen's κ

**Goal:** Quantify whether the Judge+Critic loop agrees with human labels better
than the single-prompt baseline.

**Steps:**
1. Extract individual numeric claims from each report
2. Hand-label each claim: `correct` / `incorrect` / `unverifiable`
3. Parse loop and baseline verdicts into the same labels
4. Compute Cohen's κ for each

Save human labels to `reports/human_labels.csv` with columns:
`user, claim, source_value, human_label`

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Load human labels — fill this CSV after hand-labeling
# Format: user, claim, source_value, human_label, baseline_label, loop_label
labels_path = Path('../reports/human_labels.csv')

if labels_path.exists():
    labels_df = pd.read_csv(labels_path)
    human    = labels_df['human_label'].tolist()
    baseline = labels_df['baseline_label'].tolist()
    loop     = labels_df['loop_label'].tolist()

    k_baseline = cohen_kappa_score(human, baseline)
    k_loop     = cohen_kappa_score(human, loop)

    print(f'Claims evaluated : {len(human)}')
    print(f'Baseline κ       : {k_baseline:.3f}')
    print(f'Loop κ           : {k_loop:.3f}')
    print(f'Delta            : {k_loop - k_baseline:+.3f}')
    print()
    if k_loop > k_baseline:
        print('H3 supported — Judge+Critic loop agrees with humans more than baseline')
    else:
        print('H3 not supported — loop does not improve over baseline')
    print('Report both numbers regardless of direction.')
else:
    print(f'human_labels.csv not found at {labels_path}')
    print('Complete hand-labeling first, then re-run this cell.')
